# RFM Customer Segmentation Analysis

## Project Overview

This project analyzes e-commerce customer purchasing behaviour using RFM (Recency, Frequency, Monetary) analysis.

The objective is to identify customer segments based on their purchasing behaviour and 
provide actionable insights that can support customer retention, re-engagement, and targeted marketing strategies.

### Business Objectives

- Identify high-value and loyal customers
- Identify customers who are at risk of becoming inactive
- Identify potential loyal customers and opportunities for customer development
- Understand customer contribution to overall revenue
- Develop actionable strategies for different customer segments

### Dataset

The analysis uses the **UCI Online Retail II** dataset, containing e-commerce transactions from 2009 to 2011.

### Tools & Technologies

- Python
- Pandas
- Matplotlib / Seaborn
- Jupyter Notebook
- Power BI

## 1. Import Libraries and Load Data

The dataset is provided in two Excel sheets covering the periods 2009–2010 and 2010–2011. Both sheets are loaded separately and then combined into a single dataset for analysis.

In [1]:
import pandas as pd

In [2]:
file_path = "C:/Users/disha/OneDrive/Documents/RMF customer segment case study/raw data/online_retail_II.xlsx"

df_2009_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

In [3]:
df_2009_2010.shape

(525461, 8)

In [4]:
df_2010_2011 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

In [5]:
df_2010_2011.shape

(541910, 8)

In [6]:
df_2009_2010.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

In [7]:
df_2010_2011.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

### Combining the Datasets

The two yearly datasets are combined into a single DataFrame to create a complete transaction-level dataset covering the full analysis period.

In [8]:
df=pd.concat([df_2009_2010,df_2010_2011],ignore_index=True)

## 2. Initial Data Exploration

Before performing the analysis, the combined dataset is inspected to understand its size, structure, duplicate records, and potential data-quality issues.

In [9]:
df.shape

(1067371, 8)

In [10]:
df.duplicated().sum()

34335

In [11]:
df.drop_duplicates().shape

(1033036, 8)

In [12]:
df=df.drop_duplicates()

In [13]:
df.shape

(1033036, 8)

## 3. Data Cleaning

The raw transaction data contains several data-quality issues that need to be addressed before calculating sales metrics and performing RFM analysis.

The cleaning process includes:

- Removing duplicate transactions
- Identifying and handling negative quantities associated with cancellations/returns
- Removing records with negative prices
- Identifying non-product transactions such as postage, carriage, manual charges, and adjustments
- Handling missing customer information

### 3.1 Identifying Cancelled and Returned Transactions

Negative quantities indicate transactions where products were returned or orders were cancelled. These records are identified before being removed from the dataset used for customer and revenue analysis.

In [14]:
df["Quantity"].lt(0).sum()


22496

In [15]:
df.loc[df["Quantity"]<0,"Invoice"].head()

178    C489449
179    C489449
180    C489449
181    C489449
182    C489449
Name: Invoice, dtype: object

In [16]:
df.loc[df["Quantity"]<0,"Invoice"].astype(str).str.startswith("C").sum()

19103

### 3.2 Identifying Non-Cancellation Negative Quantity Transactions

Although most negative quantity transactions are associated with cancelled invoices, some negative quantity records may not follow this pattern. These transactions are reviewed separately before removing all negative quantity records.

In [17]:
negative_non_c = df[
    (df["Quantity"] < 0) &
    (~df["Invoice"].astype(str).str.startswith("C"))
]

negative_non_c[
    ["Invoice", "StockCode", "Description", "Quantity",
     "InvoiceDate", "Price", "Customer ID"]
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN


### 3.3 Removing Negative Quantity Transactions

Negative quantity transactions represent returns or cancelled transactions and are excluded from the customer-level sales analysis.

Only transactions with a positive quantity are retained for further analysis.

In [18]:
df=df[df["Quantity"]>0]

### 3.4 Validating the Cleaned Quantity Data

The dataset is checked again to confirm that no negative quantity transactions remain after cleaning.

In [19]:
df["Quantity"].lt(0).sum()

0

### 3.5 Identifying Negative Prices
Transaction records with negative prices do not represent valid sales and can distort revenue calculations. These records are identified before being removed from the dataset.

In [20]:
df["Price"].lt(0).sum()


5

### 3.6 Removing Negative Price Transactions

Transactions with negative prices are removed because they do not represent valid sales and could distort revenue calculations.

In [21]:
df = df[df["Price"] >= 0]

### 3.7 Validating the Cleaned Price Data

The cleaned dataset is checked to confirm that no transactions with negative prices remain.

In [22]:
df["Price"].lt(0).sum()

0

### 3.8 Handling Missing Customer IDs

Customer ID is required for customer-level RFM analysis. Records without a Customer ID cannot be assigned to an individual customer, so the extent of missing customer information is assessed before calculating customer-level metrics.

In [23]:
df.shape

(1010535, 8)

In [24]:
df["Customer ID"].isna().sum()

231040

In [25]:
df["Customer ID"].isna().mean()*100

22.863136853251

In [26]:
df["Sales"]=df["Quantity"]*df["Price"]

In [27]:
df.loc[df["Customer ID"].isna(),"Sales"].sum()

3101829.7499999995

### 3.9 Removing Transactions with Missing Customer IDs

Customer-level RFM analysis requires a valid Customer ID to calculate Recency, Frequency, and Monetary values for each customer.

Although transactions with missing Customer IDs contribute to overall sales, they cannot be attributed to individual customers. Therefore, these records are excluded from the customer-level RFM analysis.

In [28]:
rfm_data=df[df["Customer ID"].notna()].copy()

In [29]:
rfm_data.shape

(779495, 9)

### 3.10 Validating the Cleaned RFM Dataset

Before calculating RFM metrics, the cleaned transaction dataset is validated for missing customer ID, invoice dates, invoice numbers, and sales values, as well as any remaining negative sales.

In [30]:
rfm_data["Customer ID"].isna().sum()

0

In [31]:
rfm_data["InvoiceDate"].isna().sum()

0

In [32]:
rfm_data["Invoice"].isna().sum()

0

In [33]:
rfm_data["Sales"].isna().sum()

0

In [34]:
(rfm_data["Sales"]<0).sum()

0

In [35]:
(rfm_data["InvoiceDate"]).min()

Timestamp('2009-12-01 07:45:00')

In [36]:
(rfm_data["InvoiceDate"]).max()

Timestamp('2011-12-09 12:50:00')

## 4. RFM Analysis

RFM analysis is used to evaluate customer purchasing behaviour using three key metrics:

- **Recency:** Number of days since the customer's most recent purchase. Lower values indicate more recent activity.
- **Frequency:** Number of unique invoices/orders placed by the customer. Higher values indicate more frequent purchasing.
- **Monetary:** Total revenue generated by the customer. Higher values indicate greater customer value.

The reference date is set to one day after the latest transaction date in the cleaned dataset. This ensures that Recency is calculated consistently based on the most recent available transaction.

In [37]:
reference_date = rfm_data["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

In [38]:
reference_date

Timestamp('2011-12-10 00:00:00')

### 4.1 Recency
Recency measures the number of days since each customer's most recent purchase. It is calculated as the difference between the reference date and the customer's latest transaction date.

Lower Recency values indicate more recent customer activity.

In [39]:
recency=rfm_data.groupby("Customer ID")["InvoiceDate"].max().reset_index()

In [40]:
recency.columns = ["Customer ID", "LastPurchaseDate"]

In [41]:
recency["Recency"]=(reference_date-recency["LastPurchaseDate"]).dt.days

In [42]:
recency.head()

,Customer ID,LastPurchaseDate,Recency
0,12346.0,2011-01-18 10:01:00,325
1,12347.0,2011-12-07 15:52:00,2
2,12348.0,2011-09-25 13:13:00,75
3,12349.0,2011-11-21 09:51:00,18
4,12350.0,2011-02-02 16:01:00,310


### 4.2 Frequency
Frequency measures the number of unique invoices or orders placed by each customer during the analysis period.

Higher Frequency values indicate that a customer purchased more frequently.

In [43]:
frequency=(rfm_data.groupby("Customer ID")["Invoice"]
           .nunique()
           .reset_index()
          )

In [44]:
frequency.columns = ["Customer ID", "Frequency"]

In [45]:
frequency.head()

,Customer ID,Frequency
0,12346.0,12
1,12347.0,8
2,12348.0,5
3,12349.0,4
4,12350.0,1


### 4.3 Monetary
Monetary measures the total sales value generated by each customer during the analysis period.

Higher Monetary values indicate greater customer contribution to revenue.

In [46]:
monetary=(
    rfm_data.groupby("Customer ID")["Sales"]
          .sum()
          .reset_index()
         )

In [47]:
monetary.columns=["Customer ID", "Monetary"]

In [48]:
monetary.head()

,Customer ID,Monetary
0,12346.0,77556.46
1,12347.0,4921.53
2,12348.0,2019.40
3,12349.0,4428.69
4,12350.0,334.40


### 4.4 Combining and Validating RFM Metrics

The three RFM metrics are combined at the customer level and checked for missing values and summary statistics before proceeding to RFM scoring.

In [49]:
rfm=recency[["Customer ID","Recency"]].merge(
    frequency,
    on="Customer ID"
).merge(
    monetary,
    on="Customer ID"
)

In [50]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [51]:
rfm.shape

(5881, 4)

In [52]:
rfm.isna().sum()

Customer ID    0
Recency        0
Frequency      0
Monetary       0
dtype: int64

The validation confirms that each customer has a complete set of Recency, Frequency, and Monetary values before scoring.

In [53]:
rfm[["Recency","Frequency","Monetary"]].describe()

,Recency,Frequency,Monetary
count,5881.000000,5881.000000,5881.000000
mean,200.992858,6.287196,2954.396237
std,209.489651,13.012879,14437.322635
min,0.000000,1.000000,0.000000
25%,25.000000,1.000000,341.900000
50%,95.000000,3.000000,865.600000
75%,379.000000,7.000000,2247.720000
max,738.000000,398.000000,580987.040000


## 5. RFM Scoring

To make the RFM metrics comparable and suitable for customer segmentation, each customer is assigned a score from **1 to 5** for Recency, Frequency, and Monetary Value.

* **Recency (R):** Lower values are better because customers who purchased more recently are more engaged.
* **Frequency (F):** Higher values are better because frequent purchases indicate stronger customer engagement.
* **Monetary (M):** Higher values are better because they indicate greater customer value.

The customers are divided into **quintiles**, assigning scores from 1 to 5 for each RFM metric.

For Recency, the scoring direction is reversed so that:

* **R = 5** represents the most recent customers.
* **R = 1** represents customers who have not purchased for the longest time.

For Frequency and Monetary:

* **5** represents the highest-value customers.
* **1** represents the lowest-value customers.

The three individual scores are then combined to create a three-digit **RFM Score**, which is used as the basis for customer segmentation.


In [54]:
rfm["R_score"]=pd.qcut(
    rfm["Recency"],
    5,
    labels=[5,4,3,2,1]
)

In [55]:
rfm["F_score"]=pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

In [56]:
rfm["M_score"]=pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

In [57]:
print("Recency Score Distribution:")
print(rfm["R_score"].value_counts().sort_index())

print("\nFrequency Score Distribution:")
print(rfm["F_score"].value_counts().sort_index())

print("\nMonetary Score Distribution:")
print(rfm["M_score"].value_counts().sort_index())

Recency Score Distribution:
R_score
5    1216
4    1140
3    1180
2    1175
1    1170
Name: count, dtype: int64

Frequency Score Distribution:
F_score
1    1177
2    1176
3    1176
4    1176
5    1176
Name: count, dtype: int64

Monetary Score Distribution:
M_score
1    1177
2    1176
3    1176
4    1176
5    1176
Name: count, dtype: int64


### 5.1 Validating RFM Scores

The assigned RFM scores are validated to ensure that each RFM metric contains scores between 1 and 5.

In [58]:
rfm[["R_score", "F_score", "M_score"]].agg(["min", "max"])

,R_score,F_score,M_score
min,5,1,1
max,1,5,5


### 5.2 Creating RFM Segments 

The individual Recency, Frequency, and Monetary scores are combined into a three-digit RFM code. Each digit represents the customer's score for Recency, Frequency, and Monetary value respectively.

For example, an RFM code of `344` indicates:
- Recency Score = 3
- Frequency Score = 4
- Monetary Score = 4

In [59]:
rfm["RFM_Segment"]= (
    rfm["R_score"].astype(str)
    +rfm["F_score"].astype(str)
    +rfm["M_score"].astype(str)
)

In [60]:
rfm[
    [
        "Customer ID",
        "R_score",
        "F_score",
        "M_score",
        "RFM_Segment"
    ]
].head(10)

,Customer ID,R_score,F_score,M_score,RFM_Segment
0,12346.0,2,5,5,255
1,12347.0,5,4,5,545
2,12348.0,3,4,4,344
3,12349.0,5,3,5,535
4,12350.0,2,1,2,212
5,12351.0,2,1,2,212
6,12352.0,4,5,4,454
7,12353.0,2,2,2,222
8,12354.0,2,1,3,213
9,12355.0,2,2,3,223


### 5.3 Calculating the Overall RFM Score

The overall RFM score is calculated by adding the Recency, Frequency, and Monetary scores for each customer.

A higher total score indicates stronger overall customer engagement across the three RFM dimensions.

In [61]:
rfm["RFM_Score"]=(
    rfm["R_score"].astype(int)
    +rfm["F_score"].astype(int)
    +rfm["M_score"].astype(int)
)

In [62]:
rfm[
    [
        "Customer ID",
        "R_score",
        "F_score",
        "M_score",
        "RFM_Score"
    ]
].head(10)


,Customer ID,R_score,F_score,M_score,RFM_Score
0,12346.0,2,5,5,12
1,12347.0,5,4,5,14
2,12348.0,3,4,4,11
3,12349.0,5,3,5,13
4,12350.0,2,1,2,5
5,12351.0,2,1,2,5
6,12352.0,4,5,4,13
7,12353.0,2,2,2,6
8,12354.0,2,1,3,6
9,12355.0,2,2,3,7


### 5.4 Validating the Overall RFM Score

The overall RFM score is checked to confirm that it falls within the expected range of 3 to 15, since each individual RFM score ranges from 1 to 5.

In [63]:
rfm["RFM_Score"].min(), rfm["RFM_Score"].max()

(3, 15)

### 5.5 Customer Segmentation

Customer segments are created by combining Recency, Frequency, and Monetary scores into meaningful groups based on customer purchasing behaviour.

The segment conditions are evaluated in a specific order because some customers may satisfy the conditions of more than one segment. The first matching condition determines the final customer segment.

In [64]:
import numpy as np

In [65]:
rfm["R_score"]=rfm["R_score"].astype(int)
rfm["F_score"]=rfm["F_score"].astype(int)
rfm["M_score"]=rfm["M_score"].astype(int)

In [66]:
rfm["Customer_Segment"] = np.select(
    [
        # Champions
        (rfm["R_score"] >= 4) &
        (rfm["F_score"] >= 4) &
        (rfm["M_score"] >= 4),

        # Can't Lose Them
        (rfm["R_score"] <= 2) &
        (rfm["F_score"] >= 4) &
        (rfm["M_score"] >= 4),

        # At Risk
        (rfm["R_score"] <= 2) &
        (rfm["F_score"] >= 3) &
        (rfm["M_score"] >= 3),

        # Loyal Customers
        (rfm["R_score"] >= 3) &
        (rfm["F_score"] >= 3) &
        (rfm["M_score"] >= 3),

        # New Customers
        (rfm["R_score"] == 5) &
        (rfm["F_score"] == 1),

        # Potential Loyalists
        (rfm["R_score"] >= 4) &
        (rfm["F_score"] <= 3) &
        (rfm["M_score"] >= 2),

        # Hibernating
        (rfm["R_score"] <= 2) &
        (rfm["F_score"] <= 2) &
        (rfm["M_score"] <= 2)
    ],
    [
        "Champions",
        "Can't Lose Them",
        "At Risk",
        "Loyal Customers",
        "New Customers",
        "Potential Loyalists",
        "Hibernating"
    ],
    default="Needs Attention"
)

In [67]:
rfm["Customer_Segment"].value_counts()

Customer_Segment
Champions              1291
Hibernating            1280
Loyal Customers        1147
Needs Attention        1134
At Risk                 389
Potential Loyalists     359
Can't Lose Them         224
New Customers            57
Name: count, dtype: int64

## 6. Customer Segmentation Analysis

The RFM scores are used to classify customers into meaningful segments based on their purchasing behaviour.

The segment-level analysis evaluates customer count, revenue contribution, and average Recency, Frequency, and Monetary values.

### 6.1 Segment-Level Performance
This analysis compares customer segments based on customer count, revenue contribution, and average RFM metrics to understand the characteristics and business value of each segment.

In [68]:
segment_summary=(
    rfm.groupby("Customer_Segment")
    .agg(
        Customers=("Customer ID","count"),
        Total_Revenue=("Monetary","sum"),
        Avg_Recency=("Recency","mean"),
        Avg_Frequency=("Frequency","mean"),
        Avg_Monetary=("Monetary","mean")
    )
    .sort_values("Total_Revenue",ascending=False)
)

In [69]:
segment_summary["Revenue_%"]=(
    segment_summary["Total_Revenue"]
    /segment_summary["Total_Revenue"].sum()
    *100
)
segment_summary

,Customers,Total_Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary,Revenue_%
Customer_Segment,,,,,,
Champions,1291,1.183665e+07,19.268009,17.140201,9168.588493,68.125359
Loyal Customers,1147,2.592613e+06,73.147341,5.966870,2260.342324,14.921680
Can't Lose Them,224,9.842864e+05,341.500000,8.955357,4394.135812,5.665022
Needs Attention,1134,7.023874e+05,215.048501,1.985891,619.389236,4.042563
At Risk,389,5.183352e+05,370.467866,3.712082,1332.481337,2.983258
Potential Loyalists,359,3.979019e+05,27.520891,2.027855,1108.361783,2.290109
Hibernating,1280,3.227322e+05,467.437500,1.184375,252.134548,1.857473
New Customers,57,1.990072e+04,10.368421,1.000000,349.135439,0.114538


### 6.2 Reactivation Opportunity

Customers in the **Can't Lose Them** and **At Risk** segments are analysed to identify customers who have historically contributed revenue but show signs of reduced recent activity.

This analysis estimates their customer count and historical revenue contribution to highlight potential reactivation opportunities.

In [70]:
reactivation_segments=["Can't Lose Them","At Risk"]

reactivation_revenue=segment_summary.loc[
    reactivation_segments,
    "Total_Revenue"
].sum()

reactivation_revenue_pct=segment_summary.loc[
    reactivation_segments,
    "Revenue_%"
].sum()

reactivation_customers=segment_summary.loc[
    reactivation_segments,
    "Customers"
].sum()

print("Customers:",reactivation_customers)
print(f"Historical Revenue: £{reactivation_revenue:,.2f}")
print(f"Revenue Share: {reactivation_revenue_pct:.2f}%")

Customers: 613
Historical Revenue: £1,502,621.66
Revenue Share: 8.65%


### 6.3 Reactivation Customer Prioritisation

Customers identified as reactivation opportunities are prioritised based on their historical monetary value. Purchase frequency is also retained as a supporting metric for evaluating customer value.

In [71]:
priority_customers=rfm[
    rfm["Customer_Segment"].isin(["Can't Lose Them","At Risk"])
].copy()

In [72]:
priority_customers=priority_customers.sort_values(
    "Monetary",ascending=False)

In [73]:
priority_customers[
    [
        "Customer ID",
        "Customer_Segment",
        "Recency",
        "Frequency",
        "Monetary",
        "R_score",
        "F_score",
        "M_score"
    ]
].head(20)

,Customer ID,Customer_Segment,Recency,Frequency,Monetary,R_score,F_score,M_score
0,12346.0,Can't Lose Them,325,12,77556.460,2,5,5
4367,16754.0,Can't Lose Them,372,29,65500.070,2,5,5
743,13093.0,Can't Lose Them,275,55,54144.250,2,5,5
5445,17850.0,Can't Lose Them,372,155,51208.870,2,5,5
3374,15749.0,At Risk,235,3,44534.300,2,3,5
1539,13902.0,Can't Lose Them,632,5,34095.260,1,4,5
135,12482.0,Can't Lose Them,576,29,23691.400,1,5,5
1699,14063.0,Can't Lose Them,438,9,22710.200,1,5,5
3432,15808.0,Can't Lose Them,306,21,17180.250,2,5,5
5051,17448.0,Can't Lose Them,497,46,14523.670,1,5,5


In [74]:
priority_customers[
    ["Recency", "Frequency", "Monetary"]
].describe().round(2)

,Recency,Frequency,Monetary
count,613.00,613.00,613.00
mean,359.88,5.63,2451.26
std,111.47,7.49,5779.41
min,190.00,2.00,609.57
25%,260.00,3.00,919.57
50%,369.00,4.00,1276.66
75%,425.00,6.00,2091.52
max,691.00,155.00,77556.46


### 6.4 Marketing Priority Classification

The 75th percentile of Monetary value and Frequency is used as a threshold to classify reactivation customers into High, Medium, and Low marketing priority groups.

In [75]:
monetary_threshold=priority_customers["Monetary"].quantile(0.75)
frequency_threshold=priority_customers["Frequency"].quantile(0.75)

priority_customers["Marketing_Priority"]=np.select(
    [
        (priority_customers["Monetary"]>=monetary_threshold) &
        (priority_customers["Frequency"]>=frequency_threshold)
    ,
        (priority_customers["Monetary"]>=monetary_threshold) |
        (priority_customers["Frequency"]>=frequency_threshold)
    ]
,
    ["High",
     "Medium"
    ],
    default="Low"
)

In [76]:
priority_customers["Marketing_Priority"].value_counts()

Marketing_Priority
Low       369
Medium    148
High       96
Name: count, dtype: int64

The resulting priority groups are reviewed to understand how many reactivation customers fall into each marketing priority level.

In [77]:
priority_summary=(
    priority_customers
    .groupby("Marketing_Priority").agg
    (
        Customers=("Customer ID","count"),
        Historical_Revenue=("Monetary","sum"),
        Avg_Recency=("Recency","mean"),
        Avg_Frequency=("Frequency","mean"),
        Avg_Monetary=("Monetary","mean")
    )
    .sort_values("Historical_Revenue",ascending=False)
)

priority_summary

,Customers,Historical_Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary
Marketing_Priority,,,,,
High,96,694270.032,335.083333,13.166667,7231.979500
Low,369,408116.510,369.756098,3.569106,1106.006802
Medium,148,400235.120,351.351351,5.871622,2704.291351


In [78]:
high_priority=priority_customers[
    priority_customers["Marketing_Priority"]=="High"
].sort_values(
    "Monetary",
    ascending=False
)


In [79]:
high_priority[
[
    "Customer ID",
    "Customer_Segment",
    "Recency",
    "Frequency",
    "Monetary",
    "R_score",
    "F_score",
    "M_score",
    "Marketing_Priority"
]
].head(20).round(2)

,Customer ID,Customer_Segment,Recency,Frequency,Monetary,R_score,F_score,M_score,Marketing_Priority
0,12346.0,Can't Lose Them,325,12,77556.46,2,5,5,High
4367,16754.0,Can't Lose Them,372,29,65500.07,2,5,5,High
743,13093.0,Can't Lose Them,275,55,54144.25,2,5,5,High
5445,17850.0,Can't Lose Them,372,155,51208.87,2,5,5,High
135,12482.0,Can't Lose Them,576,29,23691.40,1,5,5,High
1699,14063.0,Can't Lose Them,438,9,22710.20,1,5,5,High
3432,15808.0,Can't Lose Them,306,21,17180.25,2,5,5,High
5051,17448.0,Can't Lose Them,497,46,14523.67,1,5,5,High
1588,13952.0,Can't Lose Them,217,14,12823.52,2,5,5,High
1769,14134.0,Can't Lose Them,382,15,11123.35,2,5,5,High


### 6.5 Linking Customer Segments to Transaction Data

The customer-level RFM segments are merged back with the cleaned transaction data to enable transaction- and product-level analysis by customer segment.

This allows the purchasing behaviour of individual segments, such as Champions, to be examined at the product level.

In [80]:
rfm_transactions=rfm_data.merge(
    rfm[[
        "Customer ID","Customer_Segment"]],
    on="Customer ID",
    how="left"
)
    

In [81]:
rfm_transactions.shape

(779495, 10)

In [82]:
rfm_transactions[
    [
        "Customer ID",
        "Customer_Segment",
        "Description",
        "Quantity",
        "Price",
        "Sales"
    ]
].head()

,Customer ID,Customer_Segment,Description,Quantity,Price,Sales
0,13085.0,Loyal Customers,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,83.4
1,13085.0,Loyal Customers,PINK CHERRY LIGHTS,12,6.75,81.0
2,13085.0,Loyal Customers,WHITE CHERRY LIGHTS,12,6.75,81.0
3,13085.0,Loyal Customers,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,100.8
4,13085.0,Loyal Customers,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,30.0


In [83]:
rfm_transactions["Customer_Segment"].isna().sum()

0

The validation confirms that each cleaned transaction has been successfully linked to a customer segment.

### 6.6 Champion Product Analysis

The purchasing behaviour of Champions is analysed at the product level to identify products that generate the highest sales and products purchased in the largest quantities.

This can help identify products that are particularly relevant to high-value customers and can support targeted retention and cross-selling strategies.

In [84]:
champion_transactions = rfm_transactions[
    rfm_transactions["Customer_Segment"] == "Champions"
]

In [85]:
champion_transactions.shape

(484265, 10)

In [86]:
champion_top_sales=(
    champion_transactions
    .groupby("Description")
    .agg(
        Total_Sales=("Sales","sum"),
        Total_Quantity=("Quantity","sum"),
        Customers=("Customer ID","nunique")
    )
    .sort_values("Total_Sales",ascending=False)
    .head(10)
)

champion_top_sales

,Total_Sales,Total_Quantity,Customers
Description,,,
REGENCY CAKESTAND 3 TIER,215541.75,18907,528
WHITE HANGING HEART T-LIGHT HOLDER,167884.07,62459,542
JUMBO BAG RED RETROSPOT,104655.28,57997,430
ASSORTED COLOUR BIRD ORNAMENT,96114.29,61085,423
POSTAGE,84020.80,3635,121
PARTY BUNTING,70686.38,16200,391
Manual,67249.65,7597,194
CHILLI LIGHTS,58172.10,12503,162
BLACK RECORD COVER FRAME,54466.58,15931,109


In [87]:
champion_top_quantity=(
    champion_transactions
    .groupby("Description")
    .agg(
        Total_Quantity=("Quantity","sum"),
        Total_Sales=("Sales","sum"),
        Customers=("Customer ID","nunique")
    )
    .sort_values("Total_Quantity",ascending=False)
).head(10)

champion_top_quantity

,Total_Quantity,Total_Sales,Customers
Description,,,
WORLD WAR 2 GLIDERS ASSTD DESIGNS,62981,15034.07,219
WHITE HANGING HEART T-LIGHT HOLDER,62459,167884.07,542
ASSORTED COLOUR BIRD ORNAMENT,61085,96114.29,423
JUMBO BAG RED RETROSPOT,57997,104655.28,430
BROCADE RING PURSE,57904,11998.52,107
PACK OF 60 PINK PAISLEY CAKE CASES,35977,17506.65,341
SMALL POPCORN HOLDER,34298,25784.48,186
MINI PAINT SET VINTAGE,30958,19129.74,176
PACK OF 72 RETRO SPOT CAKE CASES,30078,14511.78,317


In [88]:
rfm_transactions["TotalSpend"] = rfm_transactions["Sales"]

### 6.7 Overall Business Performance

Key business metrics are calculated from the cleaned transaction dataset to provide an overall view of revenue, orders, customer base, product range, and average order value.

In [89]:
total_revenue=rfm_transactions["TotalSpend"].sum()
total_orders=rfm_transactions["Invoice"].nunique()
total_customers=rfm_transactions["Customer ID"].nunique()
total_products=rfm_transactions["StockCode"].nunique()
average_order_value=total_revenue/total_orders

print("Total Revenue: ",round(total_revenue,2))
print("Total Orders: ",total_orders)
print("Total Customers: ",total_customers)
print("Unique Products: ",total_products)
print("Average Order Value: ",round(average_order_value,2))


Total Revenue:  17374804.27
Total Orders:  36975
Total Customers:  5881
Unique Products:  4631
Average Order Value:  469.91


### 6.8 Monthly Revenue Trends

Monthly revenue is analysed to identify periods of higher sales activity and understand seasonal patterns in the business.

In [90]:
monthly_sales=(
    rfm_transactions
    .set_index("InvoiceDate")
    .resample("ME")["TotalSpend"]
    .sum()
    .reset_index()
)

In [91]:
monthly_sales.sort_values(
    "TotalSpend",
    ascending=False
).head(10)

,InvoiceDate,TotalSpend
11,2010-11-30,1166460.022
23,2011-11-30,1156205.610
22,2011-10-31,1035642.450
10,2010-10-31,1033112.010
21,2011-09-30,950690.202
9,2010-09-30,829013.951
3,2010-03-31,696978.471
0,2009-12-31,683504.010
17,2011-05-31,677355.150
18,2011-06-30,660046.050


In [92]:
monthly_sales["Year"]=monthly_sales["InvoiceDate"].dt.year
monthly_sales["Month"]=monthly_sales["InvoiceDate"].dt.month

In [93]:
monthly_sales["MoM_Growth_%"]=(
    monthly_sales["TotalSpend"]
    .pct_change()*100
).round(2)


### 6.9 Yearly Revenue Analysis

Annual revenue is aggregated to compare overall sales performance across the available years in the dataset.

In [94]:
yearly_sales=(
    monthly_sales
    .groupby("Year")["TotalSpend"]
    .sum()
    .reset_index()
)

yearly_sales

,Year,TotalSpend
0,2009,683504.010
1,2010,8374496.094
2,2011,8316804.164


### 6.10 Peak Sales Period Analysis

Revenue generated during September, October, and November is analysed to identify the contribution of the peak sales period to annual revenue.

In [95]:
peak_months=monthly_sales[
    monthly_sales["Month"].isin([9,10,11])
].groupby("Year")["TotalSpend"].sum()

peak_months

Year
2010    3028585.983
2011    3142538.262
Name: TotalSpend, dtype: float64

In [96]:
yearly_totals=monthly_sales.groupby("Year")["TotalSpend"].sum()

peak_share=(
    peak_months/yearly_totals.loc[peak_months.index]*100
).round(2)

peak_share

Year
2010    36.16
2011    37.79
Name: TotalSpend, dtype: float64

### 6.11 Country-Level Sales Analysis

Country-level performance is analysed using customer count, order volume, revenue contribution, and average order value to understand the geographic distribution of sales.

In [97]:
country_summary=rfm_transactions.groupby("Country").agg(
    Customers=("Customer ID","nunique"),
    Orders=("Invoice","nunique"),
    Revenue=("TotalSpend","sum"),
).sort_values("Revenue",ascending=False)

In [98]:
country_summary["Revenue_Share_%"]=(
    country_summary["Revenue"]
    /country_summary["Revenue"].sum()
    *100
).round(2)

In [99]:
country_summary["Average_Order_Value"]=(
    country_summary["Revenue"]
    /country_summary["Orders"]
).round(2)

country_summary.head(15)

,Customers,Orders,Revenue,Revenue_Share_%,Average_Order_Value
Country,,,,,
United Kingdom,5353,33546,1.438923e+07,82.82,428.94
EIRE,5,567,6.165705e+05,3.55,1087.43
Netherlands,22,229,5.540381e+05,3.19,2419.38
Germany,107,789,4.250197e+05,2.45,538.68
France,95,614,3.487690e+05,2.01,568.03
Australia,15,95,1.692835e+05,0.97,1781.93
Spain,41,154,1.083325e+05,0.62,703.46
Switzerland,22,90,1.000619e+05,0.58,1111.80
Sweden,19,104,9.151582e+04,0.53,879.96


### 6.12 Country-Level Customer Value

Customer revenue is analysed within each country to identify high-value customers and understand the distribution of customer value across geographic markets.

In [100]:
country_customer_value=(
    rfm_transactions
    .groupby(["Country","Customer ID"])
    .agg(
        Orders=("Invoice","nunique"),
        Revenue=("TotalSpend","sum")
    )
    .reset_index()
)

country_customer_value.sort_values(
    "Revenue",
    ascending=False
).head(20)

,Country,Customer ID,Orders,Revenue
5702,United Kingdom,18102.0,145,580987.04
389,Netherlands,14646.0,152,528602.52
105,EIRE,14156.0,156,313437.62
106,EIRE,14911.0,398,291420.81
5064,United Kingdom,17450.0,51,244784.25
1392,United Kingdom,13694.0,143,195640.69
5122,United Kingdom,17511.0,60,172132.87
4079,United Kingdom,16446.0,2,168472.50
4313,United Kingdom,16684.0,55,147142.77
8,Australia,12415.0,28,144458.37


### 6.13 Product-Level Sales Analysis

Product-level performance is analysed after excluding non-product transactions such as postage, manual charges, carriage, bank charges, and adjustments.

This ensures that product rankings represent actual merchandise rather than operational or administrative transactions.

In [101]:
non_product_pattern = (
    r"POSTAGE|Manual|BANK CHARGES|Adjust|CARRIAGE"
)

product_transactions = rfm_transactions[
    ~rfm_transactions["Description"]
    .str.contains(non_product_pattern, case=False, na=False)
].copy()

In [102]:
product_summary=(
    product_transactions
    .groupby("Description")
    .agg(
        Quantity=("Quantity","sum"),
        Orders=("Invoice","nunique"),
        Revenue=("TotalSpend","sum")
    )
    .sort_values("Revenue",ascending=False)
)

product_summary.head(20)

,Quantity,Orders,Revenue
Description,,,
REGENCY CAKESTAND 3 TIER,24139,3318,277656.25
WHITE HANGING HEART T-LIGHT HOLDER,91757,4888,247048.01
"PAPER CRAFT , LITTLE BIRDIE",80995,1,168469.60
JUMBO BAG RED RETROSPOT,74224,2612,134307.44
ASSORTED COLOUR BIRD ORNAMENT,78234,2652,124351.86
PARTY BUNTING,23464,2078,103283.38
MEDIUM CERAMIC TOP STORAGE JAR,77916,195,81416.73
PAPER CHAIN KIT 50'S CHRISTMAS,28380,1691,76598.18
CHILLI LIGHTS,14843,922,69084.30


In [103]:
product_summary.sort_values(
    "Orders",
    ascending=False
).head(20)

,Quantity,Orders,Revenue
Description,,,
WHITE HANGING HEART T-LIGHT HOLDER,91757,4888,247048.01
REGENCY CAKESTAND 3 TIER,24139,3318,277656.25
ASSORTED COLOUR BIRD ORNAMENT,78234,2652,124351.86
JUMBO BAG RED RETROSPOT,74224,2612,134307.44
PARTY BUNTING,23464,2078,103283.38
LUNCH BAG BLACK SKULL.,25392,1997,39904.00
LUNCH BAG SPACEBOY DESIGN,18834,1874,30131.70
REX CASH+CARRY JUMBO SHOPPER,3884,1857,3689.80
HOME BUILDING BLOCK WORD,7231,1831,42453.11


In [104]:
product_summary["Revenue_per_Order"]=(
    product_summary["Revenue"]/
    product_summary["Orders"]
).round(2)

In [105]:
product_summary[
    (product_summary["Orders"]>=100) &
    (product_summary["Revenue"]>=30000)
].sort_values(
    "Revenue",
    ascending=False
).head(20)

,Quantity,Orders,Revenue,Revenue_per_Order
Description,,,,
REGENCY CAKESTAND 3 TIER,24139,3318,277656.25,83.68
WHITE HANGING HEART T-LIGHT HOLDER,91757,4888,247048.01,50.54
JUMBO BAG RED RETROSPOT,74224,2612,134307.44,51.42
ASSORTED COLOUR BIRD ORNAMENT,78234,2652,124351.86,46.89
PARTY BUNTING,23464,2078,103283.38,49.70
MEDIUM CERAMIC TOP STORAGE JAR,77916,195,81416.73,417.52
PAPER CHAIN KIT 50'S CHRISTMAS,28380,1691,76598.18,45.30
CHILLI LIGHTS,14843,922,69084.30,74.93
JUMBO BAG STRAWBERRY,35842,1514,64127.77,42.36


In [106]:
product_summary = product_summary.sort_values(
    "Revenue",
    ascending=False
)

product_summary["Revenue_Share_%"] = (
    product_summary["Revenue"]
    / product_summary["Revenue"].sum()
    * 100
)

product_summary["Cumulative_Revenue_%"] = (
    product_summary["Revenue_Share_%"].cumsum()
)

In [107]:
products_to_80pct = product_summary[
    product_summary["Cumulative_Revenue_%"]<=80
]
products_to_80pct.shape

(1148, 6)

## 7. Business Insights and Recommendations

The RFM analysis identifies customer groups with different levels of engagement and revenue contribution. These segments can be used to develop targeted customer retention, reactivation, and growth strategies.

### 7.1 Customer Segment Strategies

| Customer Segment | Business Objective | Recommended Strategy |
|---|---|---|
| Champions | Retain high-value customers | VIP rewards, early access, exclusive offers |
| Loyal Customers | Strengthen loyalty | Loyalty programmes, personalised offers |
| Potential Loyalists | Encourage repeat purchases | Product recommendations, targeted promotions |
| At Risk | Prevent further inactivity | Win-back campaigns and personalised incentives |
| Can't Lose Them | Reactivate valuable customers | High-priority personalised re-engagement |
| New Customers | Encourage second purchase | Welcome campaigns and follow-up offers |
| Needs Attention | Increase engagement | Relevant promotions and product recommendations |
| Hibernating | Test reactivation potential | Low-cost reactivation campaigns |


### 7.2 Key Business Takeaways

- **Champions** represent customers with strong recent activity, purchase frequency, and monetary value and should be prioritised for retention.
- **Can't Lose Them** and **At Risk** customers represent potential reactivation opportunities because they have historically generated revenue but show weaker recent activity.
- **Potential Loyalists** can be targeted with strategies designed to increase purchase frequency and move them toward higher-value segments.
- **New Customers** should receive onboarding and follow-up communication to encourage repeat purchases.
- **Hibernating** customers can be approached with lower-cost reactivation campaigns to test whether they can be re-engaged.

## 8. Export Data for Power BI

In [108]:
# Export transaction-level data for Power BI
rfm_transactions.to_csv(
    "ecommerce_sales_data.csv",
    index=False
)

# Export customer-level RFM data for Power BI
rfm.to_csv(
    "customer_rfm_analysis.csv",
    index=False
)

print("Files exported successfully.")

Files exported successfully.


In [109]:
import os
os.getcwd()

'C:\\Users\\disha'

In [110]:
rfm["Customer ID"].nunique()

5881

In [111]:
print("Unique customers in RFM:", rfm["Customer ID"].nunique())
print("Unique customers in Sales:", df["Customer ID"].nunique())

Unique customers in RFM: 5881
Unique customers in Sales: 5881
